In [7]:
from pathlib import Path
import sys

# Find this project from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils" / "common.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project folder containing utils/common.py was not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent
CODE_DIR = PROJECT_ROOT / "utils"
sys.path.insert(0, str(CODE_DIR))

import importlib
import frac
importlib.reload(frac)

from common import find_project_root, load_ec_land_area, remap_land_area_to_esm, map_annual_to_gwl
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
from frac import (
    annual_area_weighted_fraction,
    calculate_fraction_spatial_changes,
    describe_window,
    save_fraction_spatial_plot_data,
)
import xarray as xr

MODEL_CONFIGS = [
    {"name": "EC-Earth3-ESM-1", "family": "EC", "member": "r1i1p1f1"},
    {"name": "GISS-E2-1-G-CC2", "family": "NASA", "member": "r1i1p1f3"},
    {"name": "UKESM1-2-LL", "family": "UKESM", "member": "r1i1p1f1"},
]
EXPERIMENTS = ("Exp1", "Exp2", "Exp4")
GWL_LEGS = {"Exp1": "ramp-up", "Exp4": "ramp-down-2c"}
VARIABLES = ("treeFrac", "shrubFrac", "grassFrac")
LAT_MIN, LAT_MAX = 60.0, 90.0

# Keep these compact derived files outside the repository and raw-data folders.
OUTPUT_DIR = PROJECT_ROOT.parent / f"{PROJECT_ROOT.name}_plot_data" / "frac"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# Use each model's Exp1 treeFrac grid as the target for remapping EC land area. The area-weighting function selects 60–90°N.
ec_land_area = load_ec_land_area(PROJECT_ROOT)
land_areas = {}
for config in MODEL_CONFIGS:
    family = config["family"]
    sample_folder = PROJECT_ROOT / "Exp1_data" / family / f"Exp1_{family}_treeFrac"
    sample_file = sorted(sample_folder.glob("*.nc"))[0]
    with xr.open_dataset(sample_file) as sample:
        land_areas[config["name"]] = remap_land_area_to_esm(
            ec_land_area, sample["lat"].load().values, sample["lon"].load().values
        )

## Spatial change analysis

Call the reusable spatial-analysis functions in `utils/frac.py` for every configured model and vegetation fraction. Exp1 and Exp4 use mapped GWL years within ±0.2°C; Exp2 uses the last 30 complete years. Monthly grid cells are day-weighted to annual means before averaging over selected years. Differences are percentage points. Add future dynamic vegetation models to `MODEL_CONFIGS` to include them.

In [9]:
GWL_SOURCE = PROJECT_ROOT / "tipmip-gwl" / "src"
if str(GWL_SOURCE) not in sys.path:
    sys.path.insert(0, str(GWL_SOURCE))
from tipmip_gwl import load_mapping

SPATIAL_CONFIGS = MODEL_CONFIGS
SPATIAL_GWL_TOLERANCE = 0.2
SPATIAL_EXP2_YEARS = 30
SPATIAL_PLOT_DATA_DIR = OUTPUT_DIR / "spatial"
SPATIAL_PLOT_DATA_DIR.mkdir(parents=True, exist_ok=True)

spatial_mappings = {
    config["name"]: {
        "Exp1": load_mapping(
            config["name"], leg="ramp-up", member=config["member"]
        ),
        "Exp4": load_mapping(
            config["name"], leg="ramp-down-2c", member=config["member"]
        ),
    }
    for config in SPATIAL_CONFIGS
}

In [10]:
# Select comparison periods:Exp1 and Exp4 use the ±0.2°C mapped-GWL window;
# Exp2 uses the final 30 complete calendar years.
# The selected years and GWL ranges are printed for review.

spatial_difference_fields, spatial_periods = calculate_fraction_spatial_changes(
    model_configs=SPATIAL_CONFIGS,
    project_root=PROJECT_ROOT,
    variables=VARIABLES,
    mappings=spatial_mappings,
    exp2_years=SPATIAL_EXP2_YEARS,
    gwl_tolerance=SPATIAL_GWL_TOLERANCE,
    lat_min=LAT_MIN,
    lat_max=LAT_MAX,
)

for config in SPATIAL_CONFIGS:
    periods = spatial_periods[config["name"]]
    describe_window(f"{config['name']} Exp1 baseline", periods["Exp1"], periods["Exp1_GWL"])
    describe_window(f"{config['name']} Exp2 late hold", periods["Exp2"])
    describe_window(f"{config['name']} Exp4 return", periods["Exp4"], periods["Exp4_GWL"])

EC-Earth3-ESM-1 treeFrac: Exp1 1850–1858 (9 years), Exp2 2421–2450 (30 years), Exp4 2074–2088 (15 years)
EC-Earth3-ESM-1 shrubFrac: Exp1 1850–1858 (9 years), Exp2 2421–2450 (30 years), Exp4 2074–2088 (15 years)
EC-Earth3-ESM-1 grassFrac: Exp1 1850–1858 (9 years), Exp2 2421–2450 (30 years), Exp4 2074–2088 (15 years)
GISS-E2-1-G-CC2 treeFrac: Exp1 1850–1854 (5 years), Exp2 2234–2263 (30 years), Exp4 2086–2100 (15 years)
GISS-E2-1-G-CC2 shrubFrac: Exp1 1850–1854 (5 years), Exp2 2234–2263 (30 years), Exp4 2086–2100 (15 years)
GISS-E2-1-G-CC2 grassFrac: Exp1 1850–1854 (5 years), Exp2 2234–2263 (30 years), Exp4 2086–2100 (15 years)
UKESM1-2-LL treeFrac: Exp1 1850–1852 (3 years), Exp2 2415–2444 (30 years), Exp4 2078–2096 (19 years)
UKESM1-2-LL shrubFrac: Exp1 1850–1852 (3 years), Exp2 2415–2444 (30 years), Exp4 2078–2096 (19 years)
UKESM1-2-LL grassFrac: Exp1 1850–1852 (3 years), Exp2 2415–2444 (30 years), Exp4 2078–2096 (19 years)
EC-Earth3-ESM-1 Exp1 baseline: 1850–1858 (9 years), mapped GW

In [11]:
# Save spatial plot data
spatial_plot_files = save_fraction_spatial_plot_data(
    spatial_difference_fields,
    spatial_periods,
    SPATIAL_CONFIGS,
    SPATIAL_PLOT_DATA_DIR,
    lat_min=LAT_MIN,
    lat_max=LAT_MAX,
)

Saved spatial plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/frac/spatial/EC-Earth3-ESM-1_frac_spatial_plot_data.nc
Saved spatial plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/frac/spatial/GISS-E2-1-G-CC2_frac_spatial_plot_data.nc
Saved spatial plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/frac/spatial/UKESM1-2-LL_frac_spatial_plot_data.nc
